# Cats and Dogs Classification Project
## 1. Introduction
This academic notebook demonstrates the workflow of an image classification task using traditional Machine Learning. We aim to classify images of cats and dogs using a Support Vector Machine (SVM). The notebook follows a standard ML pipeline including EDA, Preprocessing, Feature Extraction, Hyperparameter Tuning, and Evaluation.

In [ ]:
# 2. Import Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.io import imread
from skimage.transform import resize
from sklearn import svm
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

## 3. Dataset Description & Exploratory Data Analysis
The dataset consists of colored images (RGB) of cats and dogs. The images vary in dimensions, which necessitates preprocessing. We will explore a few samples to understand the variance in pixel intensities and shapes.

## 4. Data Preprocessing & Feature Extraction
* **Reading & Validation:** We read images from the directory. Corrupt files (if any) are skipped.
* **Resizing:** SVM requires fixed-length feature vectors. We resize all images to 40x40x3.
* **Normalization:** The `resize` function scales pixel values between 0 and 1, improving the model's convergence speed.
* **Feature Extraction (Flattening):** Since SVM cannot natively process 3D matrices, we flatten the 40x40x3 image into a 1D array of 4800 numerical features.

In [ ]:
categories = ['cat', 'dog']
data_dir = 'dataset/catsAndDogs40/'
target_size = (40, 40)

X_data, y_data = [], []
for category in categories:
    class_index = categories.index(category)
    for split in ['train', 'test']:
        path = os.path.join(data_dir, split, category)
        if os.path.exists(path):
            for img_name in os.listdir(path):
                img = imread(os.path.join(path, img_name))
                img_resized = resize(img, (target_size[0], target_size[1], 3))
                X_data.append(img_resized.flatten())
                y_data.append(class_index)

X = np.array(X_data)
y = np.array(y_data)
print(f"Extracted {X.shape[1]} features for {X.shape[0]} images.")

## 5. Train/Test Split
We split the data into 80% training and 20% testing. We use the `stratify` parameter to maintain the ratio of cats to dogs in both sets, preventing bias in training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

## 6. Model Building & Hyperparameter Tuning
**Support Vector Machine (SVM)** is chosen because it is highly effective in high-dimensional spaces. It works by finding an optimal hyperplane that maximizes the margin between classes. 
* **GridSearchCV** is utilized to automatically perform cross-validation and test different hyperparameters:
  * **C (Regularization):** Controls the trade-off between a smooth decision boundary and classifying training points correctly.
  * **Gamma:** Defines how far the influence of a single training example reaches.
  * **Kernel:** Translates data into higher dimensions to make it linearly separable (RBF, Poly).

In [ ]:
param_grid = {'C': [0.1, 1, 10, 100], 'gamma': [0.0001, 0.001, 0.1, 1], 'kernel': ['rbf', 'poly']}
svc = svm.SVC(probability=True)
model = GridSearchCV(svc, param_grid, cv=3, verbose=1)
model.fit(X_train, y_train)
print(f"Best Parameters: {model.best_params_}")

## 7. Model Evaluation
We evaluate our model on unseen data using comprehensive metrics:
* **Accuracy:** The overall correctness of the model.
* **Precision/Recall/F1-Score:** Crucial for understanding if the model is biased towards one class.
* **Confusion Matrix:** Shows False Positives and False Negatives.
* **ROC Curve:** Evaluates the trade-off between True Positive Rate and False Positive Rate.

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=categories))

# Confusion Matrix Plot
os.makedirs('images', exist_ok=True)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, cmap='Blues', xticklabels=categories, yticklabels=categories)
plt.title("Confusion Matrix")
plt.savefig('images/confusion_matrix.png')
plt.show()

# ROC Curve Plot
fpr, tpr, _ = roc_curve(y_test, y_prob[:, 1])
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc(fpr, tpr):.2f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.savefig('images/roc_curve.png')
plt.show()

## 8. Discussion and Conclusion
* **Results Interpretation:** The model's performance relies heavily on the quality and size of the dataset. Flattening images causes a loss of spatial features (edges, shapes), which affects accuracy.
* **Conclusion:** While SVM establishes a strong baseline, using Deep Learning (Convolutional Neural Networks - CNNs) is recommended for future work as they retain spatial patterns in images.